# P8 — Lambda-test og tredje K-datapunkt

**To spoersmaal i én kjøring:**

1. **Skaleringsformel** — bekreft K ≈ n_matriser × log₂(hidden_dim) for tre modeller
2. **Lambda-analyse** — finn relasjonen mellom P7 (spektral) og P1 (dynamisk) via tau-aggregeringer

```
K_spektral  = Σ H(W_l)          spektral entropi av vektmatriser
C0_spektral = ρ × K_spektral    ρ = 0.8625437492
tau_last    = H(kovarians av siste skjulte lag)   [bits]
tau_all     = Σ tau over alle lag                 [bits]
lambda      = C0_P1_GPT2 / C0_P7_GPT2 = ?        [skalering mellom metodene]
```

**Modeller:**
- GPT-2 (baseline, 124M)
- EleutherAI/gpt-neo-1.3B (fra P7)
- EleutherAI/gpt-neo-2.7B (ny — tredje datapunkt)

**Kjør alle celler fra topp til bunn. Bruk High-RAM runtime for 2.7B.**

In [ ]:
# CELLE 1: Installer
!pip install -q transformers torch accelerate

In [ ]:
# CELLE 2: LIMFilter
import math, time, json
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

class LIMFilter:
    def __init__(self):
        self.gamma  = 0.5772156649
        self.delta  = 4.6692016091
        self.zeta3  = 1.2020569032
        self.rho    = self.gamma / (self.delta - 4)   # 0.8625437492
        self.tau_lo = math.exp(-self.gamma)            # 0.5615
        self.tau_hi = 1.0 / self.zeta3                # 0.8319
        self.C0_P1_GPT2 = 4495.27                     # Etablert P1-baseline

    def spectral_entropy(self, W):
        if W.numel() == 0 or W.ndim < 2:
            return 0.0
        try:
            Wf = W.float()
            md = min(Wf.shape)
            if md > 2048:
                _, S, _ = torch.svd_lowrank(Wf, q=min(512, md))
            else:
                _, S, _ = torch.linalg.svd(Wf, full_matrices=False)
            s = S.detach().cpu().numpy()
            s = s[s > 1e-10]
            if len(s) == 0:
                return 0.0
            p = s / s.sum()
            return float(-(p * np.log2(p + 1e-10)).sum())
        except Exception:
            return 0.0

    def compute_K(self, model):
        K, rows = 0.0, []
        for name, p in model.named_parameters():
            if p.ndim >= 2 and p.shape[0] > 1 and p.shape[1] > 1:
                H = self.spectral_entropy(p.detach())
                if H > 0:
                    K += H
                    rows.append((H, name, list(p.shape)))
        rows.sort(reverse=True)
        return K, rows

    def tau_hidden(self, hidden_state):
        """Shannon-entropi av egenverdier til kovariansmatrisen (bits)."""
        bs, sl, hd = hidden_state.shape
        x = hidden_state.reshape(-1, hd).float()
        x = x - x.mean(0)
        L = (x.T @ x) / sl + 1e-6 * torch.eye(hd, device=x.device)
        ev = torch.linalg.eigvalsh(L)
        ev = ev[ev > 0]
        p  = ev / ev.sum()
        p  = p[p > 1e-10]
        return float(-(p * torch.log2(p)).sum())

    def tau_all_layers(self, all_hidden_states):
        """Sum av tau over alle lag inkludert embedding."""
        return sum(self.tau_hidden(h) for h in all_hidden_states)

lim = LIMFilter()
print(f"rho          = {lim.rho:.10f}")
print(f"Goldilocks   = [{lim.tau_lo:.4f}, {lim.tau_hi:.4f}]")
print(f"C0_P1_GPT2   = {lim.C0_P1_GPT2}")
print("LIMFilter klar.")

In [ ]:
# CELLE 3: Maalingsfunksjon
def measure_model(model_name, num_tau_samples=5):
    print(f"\n{'='*60}")
    print(f"MODELL: {model_name}")
    print('='*60)

    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
    )
    mdl.eval()
    cfg = mdl.config
    hd  = cfg.hidden_size
    print(f"  hidden_dim={hd}, lag={cfg.num_hidden_layers}")
    print(f"  parametere: {sum(p.numel() for p in mdl.parameters())/1e6:.1f}M")

    # --- K fra vektmatriser ---
    print("\n[1] K fra vektmatriser...")
    t0 = time.time()
    K, rows = lim.compute_K(mdl)
    n_mat = len(rows)
    C0_s  = lim.rho * K
    K_pred = n_mat * math.log2(hd)
    avvik  = abs(K - K_pred) / K * 100
    print(f"  K_spektral          = {K:.2f}")
    print(f"  C0_spektral (rho*K) = {C0_s:.2f}")
    print(f"  Matriser analysert  = {n_mat}")
    print(f"  K_pred (formel)     = {n_mat} x log2({hd}) = {K_pred:.0f}   avvik {avvik:.1f}%")
    print(f"  Tid: {time.time()-t0:.1f}s")
    print(f"  Topp 3:")
    for H, name, shape in rows[:3]:
        print(f"    {name}: H={H:.4f}, shape={shape}")

    # --- tau fra hidden states ---
    print(f"\n[2] tau fra hidden states ({num_tau_samples} samples)...")
    seed = "The coherence of a system is determined by its ability to maintain identity through constrained boundaries."
    device = next(mdl.parameters()).device
    tau_last_list, tau_all_list = [], []

    for i in range(num_tau_samples):
        inp = tok((seed * 4)[:256], return_tensors='pt', truncation=True, max_length=256)
        inp = {k: v.to(device) for k, v in inp.items()}
        with torch.no_grad():
            out = mdl(**inp, output_hidden_states=True)
        tau_last_list.append(lim.tau_hidden(out.hidden_states[-1]))
        tau_all_list.append(lim.tau_all_layers(out.hidden_states))
        if (i+1) % 2 == 0:
            print(f"  {i+1}/{num_tau_samples}: tau_last={tau_last_list[-1]:.4f}, tau_all={tau_all_list[-1]:.4f}")

    tau_last = float(np.median(tau_last_list))
    tau_all  = float(np.median(tau_all_list))
    log2_hd  = math.log2(hd)

    print(f"\n  tau_last (siste lag)        = {tau_last:.4f} bits")
    print(f"  tau_last / log2({hd})      = {tau_last/log2_hd:.4f}  [dimless]")
    print(f"  tau_all  (sum alle lag)     = {tau_all:.4f} bits")
    print(f"  tau_all  / n_lag            = {tau_all/cfg.num_hidden_layers:.4f} bits/lag")

    in_zone = lim.tau_lo <= tau_last <= lim.tau_hi
    state   = "COHERENCE" if in_zone else ("CHAOS" if tau_last < lim.tau_lo else "STASIS")
    print(f"  Goldilocks [{lim.tau_lo:.4f}, {lim.tau_hi:.4f}]: {state}")

    del mdl
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "model":      model_name,
        "hidden_dim": hd,
        "num_layers": cfg.num_hidden_layers,
        "num_matrices": n_mat,
        "K":          K,
        "K_pred":     K_pred,
        "C0_spektral": C0_s,
        "tau_last":   tau_last,
        "tau_all":    tau_all,
        "state":      state,
    }

print("Funksjon klar.")

In [ ]:
# CELLE 4: GPT-2 (baseline)
r_gpt2 = measure_model("gpt2")

In [ ]:
# CELLE 5: gpt-neo-1.3B (fra P7)
r_neo13 = measure_model("EleutherAI/gpt-neo-1.3B")

In [ ]:
# CELLE 6: gpt-neo-2.7B (ny — tredje datapunkt)
# Krever High-RAM runtime (~11GB). Bytt til det i Runtime > Change runtime type.
r_neo27 = measure_model("EleutherAI/gpt-neo-2.7B")

In [ ]:
# CELLE 7: Skaleringsformel
resultater = [r_gpt2, r_neo13, r_neo27]

print("="*70)
print("SKALERINGSFORMEL: K = n_matriser x log2(hidden_dim)")
print("="*70)
print(f"  {'Modell':<28} {'K_malt':>8} {'K_pred':>8} {'Avvik':>7} {'Matr':>6} {'C0_s':>8}")
print(f"  {'-'*68}")
for r in resultater:
    av = abs(r['K'] - r['K_pred']) / r['K'] * 100
    print(f"  {r['model']:<28} {r['K']:>8.1f} {r['K_pred']:>8.0f} {av:>6.1f}% {r['num_matrices']:>6} {r['C0_spektral']:>8.1f}")

# K-ratio mellom modellene
print(f"\n  K-ratio neo13/gpt2:  {r_neo13['K']/r_gpt2['K']:.4f}x  (hidden-ratio: {r_neo13['hidden_dim']/r_gpt2['hidden_dim']:.2f}x)")
print(f"  K-ratio neo27/gpt2:  {r_neo27['K']/r_gpt2['K']:.4f}x  (hidden-ratio: {r_neo27['hidden_dim']/r_gpt2['hidden_dim']:.2f}x)")
print(f"  K-ratio neo27/neo13: {r_neo27['K']/r_neo13['K']:.4f}x  (hidden-ratio: {r_neo27['hidden_dim']/r_neo13['hidden_dim']:.2f}x)")

print("\nKonklusjon: K er IKKE universell. K skalerer med n_matriser x log2(hidden_dim).")

In [ ]:
# CELLE 8: Lambda-analyse
C0_P1_GPT2 = lim.C0_P1_GPT2
K_P1_GPT2  = C0_P1_GPT2 / lim.rho

print("="*70)
print("LAMBDA-ANALYSE: Hva er relasjonen mellom P1 og P7?")
print("="*70)
print(f"  C0_P1_GPT2 (etablert):  {C0_P1_GPT2:.2f}")
print(f"  K_P1_GPT2  (=C0/rho):   {K_P1_GPT2:.2f}")
print(f"  C0_P7_GPT2 (malt):      {r_gpt2['C0_spektral']:.2f}")
print(f"  K_P7_GPT2  (malt):      {r_gpt2['K']:.2f}")

lambda_val = C0_P1_GPT2 / r_gpt2['C0_spektral']
print(f"\n  lambda = C0_P1 / C0_P7 = {lambda_val:.4f}")

print("\n  Kandidater for tau-aggregering som kan gi C0_P1 = 4495:")
print(f"  {'Aggregering':<40} {'Verdi GPT-2':>14} {'Ratio til C0_P1':>16}")
print(f"  {'-'*72}")

candidates = [
    ("tau_last",                                   r_gpt2['tau_last']),
    ("tau_all (sum alle lag)",                     r_gpt2['tau_all']),
    ("tau_last x seq_len(256)",                    r_gpt2['tau_last'] * 256),
    ("tau_all x seq_len(256)",                     r_gpt2['tau_all'] * 256),
    ("tau_last x hidden_dim",                      r_gpt2['tau_last'] * r_gpt2['hidden_dim']),
    ("tau_all x hidden_dim",                       r_gpt2['tau_all'] * r_gpt2['hidden_dim']),
    ("tau_last x log2(hidden_dim)^2",              r_gpt2['tau_last'] * math.log2(r_gpt2['hidden_dim'])**2),
    ("n_samples(50) x log2(hidden_dim)^2",         50 * math.log2(r_gpt2['hidden_dim'])**2),
    ("K_spektral",                                 r_gpt2['K']),
    ("K_spektral x log2(hidden_dim)",              r_gpt2['K'] * math.log2(r_gpt2['hidden_dim'])),
]

for label, val in candidates:
    ratio = C0_P1_GPT2 / val if val > 0 else float('inf')
    marker = "  <<<" if abs(ratio - 1.0) < 0.05 else ""
    print(f"  {label:<40} {val:>14.2f} {ratio:>16.4f}{marker}")

print("\n  Predikert C0_P1 for neo-1.3B og neo-2.7B (via lambda):")
for r in [r_neo13, r_neo27]:
    c0_pred = lambda_val * r['C0_spektral']
    print(f"  {r['model']}: C0_P7={r['C0_spektral']:.1f} -> C0_P1_pred={c0_pred:.1f}")

print("\nTofoo. Phi")

with open('p8_resultater.json', 'w') as f:
    json.dump({
        'gpt2': r_gpt2, 'neo13': r_neo13, 'neo27': r_neo27,
        'lambda': lambda_val,
        'C0_P1_GPT2': C0_P1_GPT2,
    }, f, indent=2)
print("Lagret: p8_resultater.json")